# 03 · 自己实现 IQL：离线强化学习为什么是榜首

IQL (Implicit Q-Learning) 的思想：从**历史日志**学"在什么状态下调多大的 alpha，长期收益最高"，
且训练时从不查询日志之外的动作——避免离线 RL 最大的坑（对没见过的动作过度乐观）。

本篇实现一个**教学版 IQL**（简化到几十行，保留三个核心组件），在官方数据构造的轨迹上训练。
硬件：CPU 即可（几分钟）。

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/caitq2024/auto_auction/blob/main/notebooks/03_%E8%87%AA%E5%B7%B1%E5%AE%9E%E7%8E%B0IQL.ipynb)

> Colab 用户先运行下面的数据下载 cell；本地运行可跳过。


In [ ]:
# Colab 环境准备：拉取教学数据（本地运行且 data/ 已存在时自动跳过）
import os, urllib.request
os.makedirs('data', exist_ok=True)
for f in ['period7_adv0.csv.gz', 'period7_tick0_adv0to7.csv.gz']:
    if not os.path.exists(f'data/{f}'):
        urllib.request.urlretrieve(f'https://github.com/caitq2024/auto_auction/raw/main/notebooks/data/{f}', f'data/{f}')
        print('downloaded', f)


In [ ]:
import pandas as pd, numpy as np, torch, torch.nn as nn
torch.manual_seed(0); np.random.seed(0)

df = pd.read_csv('data/period7_adv0.csv.gz')
BUDGET, TARGET_CPA, NUM_TICK = float(df.budget.iloc[0]), float(df.CPAConstraint.iloc[0]), 48

In [ ]:
# ---- 1. 从日志构造 (state, action, reward, next_state) 轨迹 ----
# state: [时间剩余, 预算剩余比, 近期赢单率, 近期市场价, 当期pValue均值]
# action: 该时段的有效 alpha（bid/pValue 反推）; reward: 该时段期望转化
rows = []
remaining = BUDGET
for tick, g in df.sort_values('timeStepIndex').groupby('timeStepIndex'):
    spent = g.cost[g.isExposed == 1].sum()
    state = [ (NUM_TICK-tick)/NUM_TICK, remaining/BUDGET, g.xi.mean(),
              g.leastWinningCost.mean(), g.pValue.mean()*1000 ]
    action = g.bid.mean() / max(g.pValue.mean(), 1e-9)
    reward = g[ (g.isExposed==1) ].pValue.sum()   # 期望转化
    rows.append((state, action, reward))
    remaining -= spent
states  = torch.tensor([r[0] for r in rows], dtype=torch.float32)
actions = torch.tensor([[r[1]] for r in rows], dtype=torch.float32) / 100.0  # 缩放
rewards = torch.tensor([r[2] for r in rows], dtype=torch.float32)
next_states = torch.cat([states[1:], states[-1:]])
print('轨迹长度:', len(rows))

In [ ]:
# ---- 2. 三个网络：Q(s,a) / V(s) / 策略 pi(s) ----
def mlp(i, o): return nn.Sequential(nn.Linear(i, 64), nn.ReLU(), nn.Linear(64, o))
Q, V, PI = mlp(6, 1), mlp(5, 1), mlp(5, 1)
optQ, optV, optPI = (torch.optim.Adam(m.parameters(), lr=3e-3) for m in (Q, V, PI))
TAU, GAMMA = 0.7, 0.95   # tau: expectile，IQL 的灵魂——只学到数据分布内的高分位

for step in range(2000):
    sa = torch.cat([states, actions], -1)
    # V 学 Q 的 tau 分位（expectile regression）——隐式的 max，без查询新动作
    with torch.no_grad(): q = Q(sa).squeeze()
    v = V(states).squeeze()
    diff = q - v
    lossV = (torch.abs(TAU - (diff < 0).float()) * diff**2).mean()
    optV.zero_grad(); lossV.backward(); optV.step()
    # Q 学 bellman 目标
    with torch.no_grad(): target = rewards + GAMMA * V(next_states).squeeze()
    lossQ = ((Q(sa).squeeze() - target)**2).mean()
    optQ.zero_grad(); lossQ.backward(); optQ.step()
    # 策略：advantage 加权模仿（AWR）——好动作学得多，坏动作学得少
    with torch.no_grad():
        adv = Q(sa).squeeze() - V(states).squeeze()
        w = torch.exp(3.0 * adv).clamp(max=100)
    lossPI = (w * (PI(states).squeeze() - actions.squeeze())**2).mean()
    optPI.zero_grad(); lossPI.backward(); optPI.step()
print('训练完成; 最终 lossQ/lossV/lossPI:', f'{lossQ:.4f} {lossV:.4f} {lossPI:.4f}')

In [ ]:
# ---- 3. 用 02 篇的 replay 评估（复制 replay 函数或 import）----
class IQLPolicy:
    def __init__(self): self.hist_win, self.hist_price = [], []
    def act(self, tick, remaining):
        s = torch.tensor([[(NUM_TICK-tick)/NUM_TICK, remaining/BUDGET,
                           np.mean(self.hist_win[-3:]) if self.hist_win else 0,
                           np.mean(self.hist_price[-3:]) if self.hist_price else 0.1,
                           0.5]], dtype=torch.float32)
        return float(PI(s).squeeze()) * 100.0

# （replay 函数同 02 篇，此处省略——完整可运行版见仓库 notebooks/02）
print('教学版 IQL 就绪。练习：把它接进 02 篇的 replay，和你的 PID 比一比。')

## 核心洞见

1. **expectile 回归**（`TAU=0.7`）让 V 逼近"数据里较好的结果"而不是平均——这就是"隐式"的 max；
2. **AWR 策略提取**：只对数据里出现过的动作加权模仿，从不生成分布外动作——离线安全；
3. 官方 checkpoint 的 IQL 用了完整 16 维状态 + 21 天数据，教学版只用 5 维 + 1 天——分数会差很多，但机制相同。

**练习**：把训练数据换成多个 period（下载 period-8）再训，泛化会变好吗？